In [1]:
from sedona.spark import SedonaContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [2]:
config = SedonaContext\
           .builder()\
           .master("local[*]")\
           .appName("NYTaxi")\
           .config("spark.log.level", "WARN")\
           .config("spark.driver.memory", "8g")\
           .config("spark.sql.repl.eagerEval.enabled", True)\
           .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions") \
           .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
           .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
           .config("spark.jars.packages", "org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.9.0,org.datasyslab:geotools-wrapper:1.9.0-33.5")\
           .getOrCreate()

sedona = SedonaContext.create(config)

:: loading settings :: url = jar:file:/opt/spark-3.5.8-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vadim/.ivy2/cache
The jars for the packages stored in: /home/vadim/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7aa52792-d010-4bac-af93-3c1885f9a903;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.5_2.12;1.9.0 in central
	found org.datasyslab#geotools-wrapper;1.9.0-33.5 in central
:: resolution report :: resolve 130ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.sedona#sedona-spark-shaded-3.5_2.12;1.9.0 from central in [default]
	org.datasyslab#geotools-wrapper;1.9.0-33.5 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--------------------------------------------------------------------

# Загрузка и очистка данных

Нью-Йоркская комиссия по такси и лимузинам (TLC) предоставляет подробные анонимизированные данные о поездках клиентов с 2009 года. Машины, выкрашенные жёлтым, имеют право брать пассажиров на улицах в любом из пяти боро города.

## Загружаем данные о поездках

In [3]:
raw = sedona\
        .read\
        .format("csv")\
        .option("header", "true")\
        .load("data/trips/2016")

In [4]:
print("Количество записей: ", raw.count())

[Stage 4:===============================================>         (34 + 7) / 41]

Количество записей:  34499859


In [5]:
raw

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
2,2016-02-25 17:24:20,2016-02-25 17:27:20,2,.70,-73.947250366210937,40.763771057128906,1,N,-73.992012023925781,40.735389709472656,2,5,0,0.5,0,0,0.3,5.8
2,2016-02-25 23:10:50,2016-02-25 23:31:50,2,5.52,-73.983016967773437,40.750991821289063,1,N,-73.98858642578125,40.758838653564453,2,20,0.5,0.5,0,0,0.3,21.3
2,2016-02-01 00:00:01,2016-02-01 00:10:52,6,1.99,-73.992340087890625,40.758201599121094,1,N,-73.96435546875,40.757976531982422,1,9.5,0.5,0.5,0.7,0,0.3,11.5
1,2016-02-01 00:00:04,2016-02-01 00:05:16,1,1.50,-73.981452941894531,40.749721527099609,1,N,-73.982322692871094,40.763984680175781,2,6.5,0.5,0.5,0,0,0.3,7.8
2,2016-02-01 00:00:05,2016-02-01 00:20:59,1,5.60,-74.000602722167969,40.729755401611328,1,N,-73.951324462890625,40.669834136962891,1,20,0.5,0.5,4,0,0.3,25.3
2,2016-02-01 00:00:06,2016-02-01 00:15:01,1,4.69,-74.005104064941406,40.719005584716797,1,N,-73.947090148925781,40.688884735107422,2,16,0.5,0.5,0,0,0.3,17.3
2,2016-02-01 00:00:09,2016-02-01 00:05:35,1,1.45,-73.986000061035156,40.76214599609375,1,N,-73.990608215332031,40.746208190917969,1,6.5,0.5,0.5,1.56,0,0.3,9.36
2,2016-02-01 00:00:19,2016-02-01 00:06:35,1,1.10,-73.966331481933594,40.773422241210938,1,N,-73.956108093261719,40.781421661376953,2,6.5,0.5,0.5,0,0,0.3,7.8
2,2016-02-01 00:00:21,2016-02-01 00:06:32,1,1.96,-73.976554870605469,40.765697479248047,1,N,-73.978973388671875,40.744380950927734,1,7.5,0.5,0.5,1,0,0.3,9.8
2,2016-02-01 00:00:21,2016-02-01 00:16:59,2,4.50,-73.976585388183594,40.764930725097656,1,N,-73.911117553710937,40.768909454345703,2,16,0.5,0.5,0,0,0.3,17.3


In [6]:
raw.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- pickup_longitude: string (nullable = true)
 |-- pickup_latitude: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: string (nullable = true)
 |-- dropoff_latitude: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)



In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType

tripSchema = StructType([
               StructField("VendorID", IntegerType(), nullable=True),
               StructField("tpep_pickup_datetime", TimestampType(), nullable=True),
               StructField("tpep_dropoff_datetime", TimestampType(), nullable=True),
               StructField("passenger_count", IntegerType(), nullable=True),
               StructField("trip_distance", DoubleType(), nullable=True),
               StructField("pickup_longitude", DoubleType(), nullable=True),
               StructField("pickup_latitude", DoubleType(), nullable=True),
               StructField("RatecodeID", IntegerType(), nullable=True),
               StructField("store_and_fwd_flag", StringType(), nullable=True),
               StructField("dropoff_longitude", DoubleType(), nullable=True),
               StructField("dropoff_latitude", DoubleType(), nullable=True),
               StructField("payment_type", IntegerType(), nullable=True),
               StructField("fare_amount", DoubleType(), nullable=True),
               StructField("extra", DoubleType(), nullable=True),
               StructField("mta_tax", DoubleType(), nullable=True),
               StructField("tip_amount", DoubleType(), nullable=True),
               StructField("tolls_amount", DoubleType(), nullable=True),
               StructField("improvement_surcharge", DoubleType(), nullable=True),
               StructField("total_amount", DoubleType(), nullable=True)])

In [8]:
data = sedona\
         .read\
         .format("csv")\
         .option("header", "true")\
         .schema(tripSchema)\
         .load("data/trips/2016")

In [9]:
data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



## Проверка и очистка данных

### Фильтрация данных

Удалим записи о поездках с:
- нулевой длительностью
- нулевым количеством пассажиров
- нулевым расстоянием поездки по счётчику
- координатами начала, не попадающими в прямоугольник Нью-Йорка

Нью-Йорк вписан в прямоугольник от -74.25559 до -73.70001 градусов долготы и от 40.49612 до 40.91553 широты.

In [10]:
import folium

# Создаём карту
m = folium.Map(location=[40.705825, -73.9778], zoom_start=10)

# Определяем координаты двух противоположных углов
# Формат: (широта, долгота)
top_left = [40.49612, -74.25559]
bottom_right = [40.91553, -73.70001]

# Создаём прямоугольник и добавляем его на карту
rectangle = folium.Rectangle(
    [top_left, bottom_right],  # координаты углов
    color='blue',              # цвет контура
    fill=True,                 # включить заливку
    fill_color='blue',         # цвет заливки
    fill_opacity=0.3,          # прозрачность заливки
    weight=2                   # толщина контура
)
rectangle.add_to(m)

# Сохраняем карту в HTML-файл
m.save('ny.html')

# Отображаем карту
m

In [11]:
ny_min_lon = -74.25559
ny_max_lon = -73.70001
ny_min_lat = 40.49612
ny_max_lat = 40.91553

In [12]:
trips = data\
          .filter(f.col("tpep_pickup_datetime") < f.col("tpep_dropoff_datetime"))\
          .filter(f.col("passenger_count") > 0)\
          .filter(f.col("trip_distance") > 0)\
          .filter(f.col("pickup_longitude") >= ny_min_lon)\
          .filter(f.col("pickup_longitude") <= ny_max_lon)\
          .filter(f.col("pickup_latitude") >= ny_min_lat)\
          .filter(f.col("pickup_latitude") <= ny_max_lat)\
          .filter(f.col("tpep_pickup_datetime") >= "2016-01-01 00:00:00")\
          .filter(f.col("tpep_pickup_datetime") <= "2016-03-31 23:59:59")

In [13]:
trips.count()

33798171

In [14]:
trips

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
2,2016-02-25 17:24:20,2016-02-25 17:27:20,2,0.7,-73.94725036621094,40.763771057128906,1,N,-73.99201202392578,40.735389709472656,2,5.0,0.0,0.5,0.0,0.0,0.3,5.8
2,2016-02-25 23:10:50,2016-02-25 23:31:50,2,5.52,-73.98301696777344,40.75099182128906,1,N,-73.98858642578125,40.75883865356445,2,20.0,0.5,0.5,0.0,0.0,0.3,21.3
2,2016-02-01 00:00:01,2016-02-01 00:10:52,6,1.99,-73.99234008789062,40.758201599121094,1,N,-73.96435546875,40.75797653198242,1,9.5,0.5,0.5,0.7,0.0,0.3,11.5
1,2016-02-01 00:00:04,2016-02-01 00:05:16,1,1.5,-73.98145294189453,40.74972152709961,1,N,-73.9823226928711,40.76398468017578,2,6.5,0.5,0.5,0.0,0.0,0.3,7.8
2,2016-02-01 00:00:05,2016-02-01 00:20:59,1,5.6,-74.00060272216797,40.72975540161133,1,N,-73.95132446289062,40.66983413696289,1,20.0,0.5,0.5,4.0,0.0,0.3,25.3
2,2016-02-01 00:00:06,2016-02-01 00:15:01,1,4.69,-74.0051040649414,40.7190055847168,1,N,-73.94709014892578,40.68888473510742,2,16.0,0.5,0.5,0.0,0.0,0.3,17.3
2,2016-02-01 00:00:09,2016-02-01 00:05:35,1,1.45,-73.98600006103516,40.76214599609375,1,N,-73.99060821533203,40.74620819091797,1,6.5,0.5,0.5,1.56,0.0,0.3,9.36
2,2016-02-01 00:00:19,2016-02-01 00:06:35,1,1.1,-73.9663314819336,40.77342224121094,1,N,-73.95610809326172,40.78142166137695,2,6.5,0.5,0.5,0.0,0.0,0.3,7.8
2,2016-02-01 00:00:21,2016-02-01 00:06:32,1,1.96,-73.97655487060547,40.76569747924805,1,N,-73.97897338867188,40.744380950927734,1,7.5,0.5,0.5,1.0,0.0,0.3,9.8
2,2016-02-01 00:00:21,2016-02-01 00:16:59,2,4.5,-73.9765853881836,40.764930725097656,1,N,-73.91111755371094,40.7689094543457,2,16.0,0.5,0.5,0.0,0.0,0.3,17.3


# Геопространственный анализ

## Преобразуем координаты в точки

In [15]:
from sedona.spark.sql.st_constructors import ST_Point

tripsPoints = trips\
                .withColumn("pickupPoint", ST_Point("pickup_longitude", "pickup_latitude"))\
                .withColumn("dropoffPoint", ST_Point("dropoff_longitude", "dropoff_latitude"))

In [16]:
tripsPoints.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickupPoint: geometry (nullable = true)
 |-- dropoffPoint: geometry (nullable = true)



In [17]:
tripsPoints

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,pickupPoint,dropoffPoint
2,2016-02-25 17:24:20,2016-02-25 17:27:20,2,0.7,-73.94725036621094,40.763771057128906,1,N,-73.99201202392578,40.735389709472656,2,5.0,0.0,0.5,0.0,0.0,0.3,5.8,POINT (-73.947250...,POINT (-73.992012...
2,2016-02-25 23:10:50,2016-02-25 23:31:50,2,5.52,-73.98301696777344,40.75099182128906,1,N,-73.98858642578125,40.75883865356445,2,20.0,0.5,0.5,0.0,0.0,0.3,21.3,POINT (-73.983016...,POINT (-73.988586...
2,2016-02-01 00:00:01,2016-02-01 00:10:52,6,1.99,-73.99234008789062,40.758201599121094,1,N,-73.96435546875,40.75797653198242,1,9.5,0.5,0.5,0.7,0.0,0.3,11.5,POINT (-73.992340...,POINT (-73.964355...
1,2016-02-01 00:00:04,2016-02-01 00:05:16,1,1.5,-73.98145294189453,40.74972152709961,1,N,-73.9823226928711,40.76398468017578,2,6.5,0.5,0.5,0.0,0.0,0.3,7.8,POINT (-73.981452...,POINT (-73.982322...
2,2016-02-01 00:00:05,2016-02-01 00:20:59,1,5.6,-74.00060272216797,40.72975540161133,1,N,-73.95132446289062,40.66983413696289,1,20.0,0.5,0.5,4.0,0.0,0.3,25.3,POINT (-74.000602...,POINT (-73.951324...
2,2016-02-01 00:00:06,2016-02-01 00:15:01,1,4.69,-74.0051040649414,40.7190055847168,1,N,-73.94709014892578,40.68888473510742,2,16.0,0.5,0.5,0.0,0.0,0.3,17.3,POINT (-74.005104...,POINT (-73.947090...
2,2016-02-01 00:00:09,2016-02-01 00:05:35,1,1.45,-73.98600006103516,40.76214599609375,1,N,-73.99060821533203,40.74620819091797,1,6.5,0.5,0.5,1.56,0.0,0.3,9.36,POINT (-73.986000...,POINT (-73.990608...
2,2016-02-01 00:00:19,2016-02-01 00:06:35,1,1.1,-73.9663314819336,40.77342224121094,1,N,-73.95610809326172,40.78142166137695,2,6.5,0.5,0.5,0.0,0.0,0.3,7.8,POINT (-73.966331...,POINT (-73.956108...
2,2016-02-01 00:00:21,2016-02-01 00:06:32,1,1.96,-73.97655487060547,40.76569747924805,1,N,-73.97897338867188,40.744380950927734,1,7.5,0.5,0.5,1.0,0.0,0.3,9.8,POINT (-73.976554...,POINT (-73.978973...
2,2016-02-01 00:00:21,2016-02-01 00:16:59,2,4.5,-73.9765853881836,40.764930725097656,1,N,-73.91111755371094,40.7689094543457,2,16.0,0.5,0.5,0.0,0.0,0.3,17.3,POINT (-73.976585...,POINT (-73.911117...


In [21]:
tripsData = tripsPoints\
                  .select(\
                    f.col("tpep_pickup_datetime").alias("pickupDatetime"),\
                    f.col("pickupPoint"),\
                    f.col("passenger_count"),\
                    f.col("trip_distance"),\
                    f.col("total_amount"),\
                    f.col("tpep_dropoff_datetime").alias("dropoffDatetime"),\
                    f.col("dropoffPoint"))

In [22]:
tripsData

pickupDatetime,pickupPoint,passenger_count,trip_distance,total_amount,dropoffDatetime,dropoffPoint
2016-02-25 17:24:20,POINT (-73.947250...,2,0.7,5.8,2016-02-25 17:27:20,POINT (-73.992012...
2016-02-25 23:10:50,POINT (-73.983016...,2,5.52,21.3,2016-02-25 23:31:50,POINT (-73.988586...
2016-02-01 00:00:01,POINT (-73.992340...,6,1.99,11.5,2016-02-01 00:10:52,POINT (-73.964355...
2016-02-01 00:00:04,POINT (-73.981452...,1,1.5,7.8,2016-02-01 00:05:16,POINT (-73.982322...
2016-02-01 00:00:05,POINT (-74.000602...,1,5.6,25.3,2016-02-01 00:20:59,POINT (-73.951324...
2016-02-01 00:00:06,POINT (-74.005104...,1,4.69,17.3,2016-02-01 00:15:01,POINT (-73.947090...
2016-02-01 00:00:09,POINT (-73.986000...,1,1.45,9.36,2016-02-01 00:05:35,POINT (-73.990608...
2016-02-01 00:00:19,POINT (-73.966331...,1,1.1,7.8,2016-02-01 00:06:35,POINT (-73.956108...
2016-02-01 00:00:21,POINT (-73.976554...,1,1.96,9.8,2016-02-01 00:06:32,POINT (-73.978973...
2016-02-01 00:00:21,POINT (-73.976585...,2,4.5,17.3,2016-02-01 00:16:59,POINT (-73.911117...


## Загружаем данные о зонах в GeoJSON

### Вариант 1

In [25]:
geoZones = sedona\
             .read\
             .format("geojson")\
             .option("multiLine", "true")\
             .load("data/NYC_Taxi_Zones.geojson")\
             .selectExpr("explode(features) as features")\
             .select("features.*")\
             .withColumn("objectID", f.expr("properties['objectid']").cast(IntegerType()))\
             .withColumn("locationID", f.expr("properties['location_id']").cast(IntegerType()))\
             .withColumn("zone", f.expr("properties['zone']"))\
             .withColumn("borough", f.expr("properties['borough']"))\
             .withColumn("shapeArea", f.expr("properties['shape_area']").cast(DoubleType()))\
             .withColumn("shapeLeng", f.expr("properties['shape_leng']").cast(DoubleType()))\
             .drop("properties", "type")

In [26]:
geoZones.printSchema()

root
 |-- geometry: geometry (nullable = true)
 |-- objectID: integer (nullable = true)
 |-- locationID: integer (nullable = true)
 |-- zone: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- shapeArea: double (nullable = true)
 |-- shapeLeng: double (nullable = true)



In [27]:
geoZones

geometry,objectID,locationID,zone,borough,shapeArea,shapeLeng
MULTIPOLYGON (((-...,1,1,Newark Airport,EWR,7.823067885E-4,0.116357453189
MULTIPOLYGON (((-...,2,2,Jamaica Bay,Queens,0.00486634037837,0.43346966679
MULTIPOLYGON (((-...,3,3,Allerton/Pelham G...,Bronx,3.14414156821E-4,0.0843411059012
MULTIPOLYGON (((-...,4,4,Alphabet City,Manhattan,1.11871946192E-4,0.0435665270921
MULTIPOLYGON (((-...,5,5,Arden Heights,Staten Island,4.97957489363E-4,0.0921464898574
MULTIPOLYGON (((-...,6,6,Arrochar/Fort Wad...,Staten Island,6.06460984581E-4,0.150490542523
MULTIPOLYGON (((-...,7,7,Astoria,Queens,3.89787989274E-4,0.107417171123
MULTIPOLYGON (((-...,8,8,Astoria Park,Queens,2.6587716279E-5,0.0275906911574
MULTIPOLYGON (((-...,9,9,Auburndale,Queens,3.38443803197E-4,0.0997840924705
MULTIPOLYGON (((-...,24,24,Bloomingdale,Manhattan,6.07235737749E-5,0.0469999619287


Оставляем названия зоны, боро и геометрию.

In [28]:
zones = geoZones.select("zone", "borough", "geometry").cache()

In [29]:
zones

zone,borough,geometry
Newark Airport,EWR,MULTIPOLYGON (((-...
Jamaica Bay,Queens,MULTIPOLYGON (((-...
Allerton/Pelham G...,Bronx,MULTIPOLYGON (((-...
Alphabet City,Manhattan,MULTIPOLYGON (((-...
Arden Heights,Staten Island,MULTIPOLYGON (((-...
Arrochar/Fort Wad...,Staten Island,MULTIPOLYGON (((-...
Astoria,Queens,MULTIPOLYGON (((-...
Astoria Park,Queens,MULTIPOLYGON (((-...
Auburndale,Queens,MULTIPOLYGON (((-...
Bloomingdale,Manhattan,MULTIPOLYGON (((-...


### Вариант 2

In [30]:
shapes = sedona.read.format("shapefile").load("data/zones")

In [31]:
shapes.printSchema()

root
 |-- geometry: geometry (nullable = true)
 |-- OBJECTID: long (nullable = true)
 |-- Shape_Leng: decimal(19,11) (nullable = true)
 |-- Shape_Area: decimal(19,11) (nullable = true)
 |-- zone: string (nullable = true)
 |-- LocationID: long (nullable = true)
 |-- borough: string (nullable = true)



In [32]:
shapes

geometry,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough
POLYGON ((933100....,1,0.11635745319,0.00078230679,Newark Airport,1,EWR
MULTIPOLYGON (((1...,2,0.43346966679,0.00486634038,Jamaica Bay,2,Queens
POLYGON ((1026308...,3,0.08434110590,0.00031441416,Allerton/Pelham G...,3,Bronx
POLYGON ((992073....,4,0.04356652709,0.00011187195,Alphabet City,4,Manhattan
POLYGON ((935843....,5,0.09214648986,0.00049795749,Arden Heights,5,Staten Island
POLYGON ((966568....,6,0.15049054252,0.00060646098,Arrochar/Fort Wad...,6,Staten Island
POLYGON ((1010804...,7,0.10741717112,0.00038978799,Astoria,7,Queens
POLYGON ((1005482...,8,0.02759069116,0.00002658772,Astoria Park,8,Queens
POLYGON ((1043803...,9,0.09978409247,0.00033844380,Auburndale,9,Queens
POLYGON ((1044355...,10,0.09983947942,0.00043582382,Baisley Park,10,Queens


In [38]:
zones2 = shapes.select("zone", "borough", "geometry")

In [39]:
zones2

zone,borough,geometry
Newark Airport,EWR,POLYGON ((933100....
Jamaica Bay,Queens,MULTIPOLYGON (((1...
Allerton/Pelham G...,Bronx,POLYGON ((1026308...
Alphabet City,Manhattan,POLYGON ((992073....
Arden Heights,Staten Island,POLYGON ((935843....
Arrochar/Fort Wad...,Staten Island,POLYGON ((966568....
Astoria,Queens,POLYGON ((1010804...
Astoria Park,Queens,POLYGON ((1005482...
Auburndale,Queens,POLYGON ((1043803...
Baisley Park,Queens,POLYGON ((1044355...


## Соединяем поездки с зонами

In [43]:
from sedona.spark.sql.st_predicates import ST_Contains

tripsZones = tripsData\
               .join(\
                  f.broadcast(zones),\
                  ST_Contains(zones["geometry"], tripsData["pickupPoint"]), "left")\
               .withColumnRenamed("zone", "pickupZone")\
               .withColumnRenamed("borough", "pickupBorough")\
               .drop("geometry")\
               .join(\
                  f.broadcast(zones),\
                  ST_Contains(zones["geometry"], tripsData["dropoffPoint"]), "left")\
               .withColumnRenamed("zone", "dropoffZone")\
               .withColumnRenamed("borough", "dropoffBorough")\
               .drop("geometry")\
               .na.fill("NA", ["pickupZone", "pickupBorough", "dropoffZone", "dropoffBorough"])\
               .cache()

In [44]:
tripsZones.printSchema()

root
 |-- pickupDatetime: timestamp (nullable = true)
 |-- pickupPoint: geometry (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- dropoffDatetime: timestamp (nullable = true)
 |-- dropoffPoint: geometry (nullable = true)
 |-- pickupZone: string (nullable = false)
 |-- pickupBorough: string (nullable = false)
 |-- dropoffZone: string (nullable = false)
 |-- dropoffBorough: string (nullable = false)



In [45]:
tripsZones

pickupDatetime,pickupPoint,passenger_count,trip_distance,total_amount,dropoffDatetime,dropoffPoint,pickupZone,pickupBorough,dropoffZone,dropoffBorough
2016-02-25 17:24:20,POINT (-73.947250...,2,0.7,5.8,2016-02-25 17:27:20,POINT (-73.992012...,Roosevelt Island,Manhattan,Union Sq,Manhattan
2016-02-25 23:10:50,POINT (-73.983016...,2,5.52,21.3,2016-02-25 23:31:50,POINT (-73.988586...,Midtown South,Manhattan,Times Sq/Theatre ...,Manhattan
2016-02-01 00:00:01,POINT (-73.992340...,6,1.99,11.5,2016-02-01 00:10:52,POINT (-73.964355...,Clinton East,Manhattan,Sutton Place/Turt...,Manhattan
2016-02-01 00:00:04,POINT (-73.981452...,1,1.5,7.8,2016-02-01 00:05:16,POINT (-73.982322...,Murray Hill,Manhattan,Midtown North,Manhattan
2016-02-01 00:00:05,POINT (-74.000602...,1,5.6,25.3,2016-02-01 00:20:59,POINT (-73.951324...,Greenwich Village...,Manhattan,Crown Heights South,Brooklyn
2016-02-01 00:00:06,POINT (-74.005104...,1,4.69,17.3,2016-02-01 00:15:01,POINT (-73.947090...,TriBeCa/Civic Center,Manhattan,Bedford,Brooklyn
2016-02-01 00:00:09,POINT (-73.986000...,1,1.45,9.36,2016-02-01 00:05:35,POINT (-73.990608...,Times Sq/Theatre ...,Manhattan,Penn Station/Madi...,Manhattan
2016-02-01 00:00:19,POINT (-73.966331...,1,1.1,7.8,2016-02-01 00:06:35,POINT (-73.956108...,Central Park,Manhattan,Upper East Side N...,Manhattan
2016-02-01 00:00:21,POINT (-73.976554...,1,1.96,9.8,2016-02-01 00:06:32,POINT (-73.978973...,Midtown North,Manhattan,Kips Bay,Manhattan
2016-02-01 00:00:21,POINT (-73.976585...,2,4.5,17.3,2016-02-01 00:16:59,POINT (-73.911117...,Midtown North,Manhattan,Steinway,Queens


In [79]:
rides = tripsZones.select(
          "pickupDatetime",
          "pickupZone",
          "pickupBorough",
          "dropoffDatetime",
          "dropoffZone",
          "dropoffBorough",
          "passenger_count",
          "trip_distance",
          "total_amount")

In [80]:
rides

pickupDatetime,pickupZone,pickupBorough,dropoffDatetime,dropoffZone,dropoffBorough,passenger_count,trip_distance,total_amount
2016-02-25 17:24:20,Roosevelt Island,Manhattan,2016-02-25 17:27:20,Union Sq,Manhattan,2,0.7,5.8
2016-02-25 23:10:50,Midtown South,Manhattan,2016-02-25 23:31:50,Times Sq/Theatre ...,Manhattan,2,5.52,21.3
2016-02-01 00:00:01,Clinton East,Manhattan,2016-02-01 00:10:52,Sutton Place/Turt...,Manhattan,6,1.99,11.5
2016-02-01 00:00:04,Murray Hill,Manhattan,2016-02-01 00:05:16,Midtown North,Manhattan,1,1.5,7.8
2016-02-01 00:00:05,Greenwich Village...,Manhattan,2016-02-01 00:20:59,Crown Heights South,Brooklyn,1,5.6,25.3
2016-02-01 00:00:06,TriBeCa/Civic Center,Manhattan,2016-02-01 00:15:01,Bedford,Brooklyn,1,4.69,17.3
2016-02-01 00:00:09,Times Sq/Theatre ...,Manhattan,2016-02-01 00:05:35,Penn Station/Madi...,Manhattan,1,1.45,9.36
2016-02-01 00:00:19,Central Park,Manhattan,2016-02-01 00:06:35,Upper East Side N...,Manhattan,1,1.1,7.8
2016-02-01 00:00:21,Midtown North,Manhattan,2016-02-01 00:06:32,Kips Bay,Manhattan,1,1.96,9.8
2016-02-01 00:00:21,Midtown North,Manhattan,2016-02-01 00:16:59,Steinway,Queens,2,4.5,17.3


## Агрегация данных по часам и районам посадки

In [81]:
tripsMDH = rides\
             .withColumn("month", f.month(f.col("pickupDatetime")))\
             .withColumn("day", f.dayofmonth(f.col("pickupDatetime")))\
             .withColumn("hour", f.hour(f.col("pickupDatetime")))

In [82]:
tripsMDH

pickupDatetime,pickupZone,pickupBorough,dropoffDatetime,dropoffZone,dropoffBorough,passenger_count,trip_distance,total_amount,month,day,hour
2016-02-25 17:24:20,Roosevelt Island,Manhattan,2016-02-25 17:27:20,Union Sq,Manhattan,2,0.7,5.8,2,25,17
2016-02-25 23:10:50,Midtown South,Manhattan,2016-02-25 23:31:50,Times Sq/Theatre ...,Manhattan,2,5.52,21.3,2,25,23
2016-02-01 00:00:01,Clinton East,Manhattan,2016-02-01 00:10:52,Sutton Place/Turt...,Manhattan,6,1.99,11.5,2,1,0
2016-02-01 00:00:04,Murray Hill,Manhattan,2016-02-01 00:05:16,Midtown North,Manhattan,1,1.5,7.8,2,1,0
2016-02-01 00:00:05,Greenwich Village...,Manhattan,2016-02-01 00:20:59,Crown Heights South,Brooklyn,1,5.6,25.3,2,1,0
2016-02-01 00:00:06,TriBeCa/Civic Center,Manhattan,2016-02-01 00:15:01,Bedford,Brooklyn,1,4.69,17.3,2,1,0
2016-02-01 00:00:09,Times Sq/Theatre ...,Manhattan,2016-02-01 00:05:35,Penn Station/Madi...,Manhattan,1,1.45,9.36,2,1,0
2016-02-01 00:00:19,Central Park,Manhattan,2016-02-01 00:06:35,Upper East Side N...,Manhattan,1,1.1,7.8,2,1,0
2016-02-01 00:00:21,Midtown North,Manhattan,2016-02-01 00:06:32,Kips Bay,Manhattan,1,1.96,9.8,2,1,0
2016-02-01 00:00:21,Midtown North,Manhattan,2016-02-01 00:16:59,Steinway,Queens,2,4.5,17.3,2,1,0


In [85]:
tripsAgg = tripsMDH\
             .groupBy("month", "day", "hour", "pickupZone")\
             .agg(\
                f.count("*").alias("trips"),\
                f.avg("trip_distance").alias("avg_distance"),\
                f.avg("total_amount").alias("avg_amount"))\
             .orderBy("month", "day", "hour", "pickupZone")\
             .cache()

In [86]:
tripsAgg

month,day,hour,pickupZone,trips,avg_distance,avg_amount
1,1,0,Alphabet City,153,2.8382352941176476,14.64058823529411
1,1,0,Astoria,113,2.8107079646017707,13.155044247787597
1,1,0,Astoria Park,1,1.67,11.16
1,1,0,Battery Park,9,3.7866666666666666,16.183333333333334
1,1,0,Battery Park City,131,3.8383206106870236,17.92351145038167
1,1,0,Bay Ridge,7,4.1171428571428565,16.922857142857143
1,1,0,Bedford,36,3.534722222222222,16.524166666666666
1,1,0,Bedford Park,2,3.45,13.3
1,1,0,Bensonhurst East,1,3.93,18.8
1,1,0,Bloomingdale,110,3.4789090909090894,16.42772727272726


## Заполнение пропусков

Загрузим список зон и создадим набор данных с записями со всеми зонами на каждый час каждого дня с января по март.

In [111]:
hours = list(range(0, 24))
days = list(range(1, 32))
months = list(range(1, 4))

reference = sedona\
              .read\
              .format("csv")\
              .option("header", "true")\
              .option("inferSchema", "true")\
              .load("data/taxi_zone_lookup.csv")\
              .select(f.col("Borough").alias("pickupBorough"), f.col("Zone").alias("pickupZone"))\
              .withColumn("hour", f.explode(f.lit(hours)))\
              .withColumn("day", f.explode(f.lit(days)))\
              .withColumn("month", f.explode(f.lit(months)))\
              .filter("NOT (month == 2 AND day > 29)")\
              .select("month", "day", "hour", "pickupBorough", "pickupZone")\
              .orderBy("month", "day", "hour", "pickupBorough", "pickupZone")

In [113]:
reference

month,day,hour,pickupBorough,pickupZone
1,1,0,Bronx,Allerton/Pelham G...
1,1,0,Bronx,Bedford Park
1,1,0,Bronx,Belmont
1,1,0,Bronx,Bronx Park
1,1,0,Bronx,Bronxdale
1,1,0,Bronx,City Island
1,1,0,Bronx,Claremont/Bathgate
1,1,0,Bronx,Co-Op City
1,1,0,Bronx,Country Club
1,1,0,Bronx,Crotona Park


In [114]:
data = reference\
         .join(tripsAgg, ["month", "day", "hour", "pickupZone"], "leftouter")\
         .na\
         .fill(0, ["trips", "avg_distance", "avg_amount"])

In [115]:
data.orderBy("month", "day", "hour", "pickupZone", "pickupBorough")

month,day,hour,pickupZone,pickupBorough,trips,avg_distance,avg_amount
1,1,0,Allerton/Pelham G...,Bronx,0,0.0,0.0
1,1,0,Alphabet City,Manhattan,153,2.8382352941176476,14.64058823529411
1,1,0,Arden Heights,Staten Island,0,0.0,0.0
1,1,0,Arrochar/Fort Wad...,Staten Island,0,0.0,0.0
1,1,0,Astoria,Queens,113,2.8107079646017707,13.155044247787597
1,1,0,Astoria Park,Queens,1,1.67,11.16
1,1,0,Auburndale,Queens,0,0.0,0.0
1,1,0,Baisley Park,Queens,0,0.0,0.0
1,1,0,Bath Beach,Brooklyn,0,0.0,0.0
1,1,0,Battery Park,Manhattan,9,3.7866666666666666,16.183333333333334
